# 04 - Train / validation / test split (RQ1)

Splits the labelled corpus into training, validation and held-out test sets for
classifier development. Reads `data/processed/labelled_comments_final.csv` and
writes the splits to `data/processed/`.

- **Train:** 1,620 comments (split into train-fit and validation below)
- **Train-fit:** 1,215 comments (model fitting)
- **Validation:** 405 comments (checkpoint and threshold selection)
- **Test:** 406 comments (held out; used only for final evaluation in notebook 10)

All splits use seed 42. The test set is never seen during development.

In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
proc = PROJECT_ROOT / "data" / "processed"

label_cols = ["label_1A", "label_1B", "label_2"]

labelled = pd.read_csv(proc / "labelled_comments_final.csv")

# Reliable provenance: stratified rows carry a sample_source; keyword-enrichment rows don't
labelled["sample_stage"] = labelled["sample_source"].notna().map(
    {True: "stratified", False: "keyword_enrichment"}
)

labelled = labelled.drop_duplicates("text_clean").reset_index(drop=True)
labelled.to_csv(proc / "labelled_comments_final.csv", index=False)
print("total:", len(labelled))
print(labelled["sample_stage"].value_counts())

# Work from a single dataframe in the cells below
df = labelled

In [ ]:
# De-duplicate on exact cleaned text (already applied above; kept as a safeguard)
before = len(df)
df = df.drop_duplicates(subset="text_clean", keep="first").reset_index(drop=True)
print(f"dropped {before - len(df)} duplicate texts, {len(df)} remain")

In [ ]:
# Build stratification key 
# Concatenates labels into one value 

strat = df[label_cols].astype(str).agg("".join, axis=1)
print(strat.value_counts())

In [ ]:
# Split

train, test = train_test_split(
    df,
    test_size=0.20,
    stratify=strat, # stratify=strat keeps the proportion of each label combination the same in train and test.
    random_state=42,
)

In [ ]:
# Print number of positives under each construct and save the test / train splits to data folder

print(f"train {len(train)}  test {len(test)}")
print("\ntrain positives:\n", train[label_cols].sum())
print("\ntest positives:\n", test[label_cols].sum())
print("\ntrain rate:\n", train[label_cols].mean().round(3))
print("test rate:\n", test[label_cols].mean().round(3))

train.to_csv(proc / "train.csv", index=False)
test.to_csv(proc / "test.csv", index=False)

In [ ]:
train[label_cols].mean().round(3)
test[label_cols].mean().round(3)

In [ ]:
# Confirm that no exact text appears in both sets
assert set(train["text_clean"]).isdisjoint(set(test["text_clean"]))

# Confirm all labels are complete and binary
assert train[label_cols].notna().all().all()
assert test[label_cols].notna().all().all()

for col in label_cols:
    assert set(df[col].unique()).issubset({0, 1})

# Check keyword/random sampling balance
print("\nTrain sample source:")
print(train["sample_source"].value_counts(normalize=True).round(3))

print("\nTest sample source:")
print(test["sample_source"].value_counts(normalize=True).round(3))

In [ ]:
# Build stratification key for the current training set

strat_train = train[label_cols].astype(str).agg("".join, axis=1)

print("Training label combinations before validation split:")
print(strat_train.value_counts())

# Split the current training data into:
# train_fit = data used to fit models
# val = data used to compare and improve models

train_fit, val = train_test_split(
    train,
    test_size=0.25,
    stratify=strat_train,
    random_state=42,
)

# Reset row numbers after splitting
train_fit = train_fit.reset_index(drop=True)
val = val.reset_index(drop=True)

print(f"\ntrain_fit: {len(train_fit)}")
print(f"validation: {len(val)}")
print(f"final test: {len(test)}")

print("\nTrain-fit label rates:")
print(train_fit[label_cols].mean().round(3))

print("\nValidation label rates:")
print(val[label_cols].mean().round(3))

In [ ]:
# Check the actual number of positive examples in each split

print("Train-fit positives:")
print(train_fit[label_cols].sum())

print("\nValidation positives:")
print(val[label_cols].sum())

print("\nFinal-test positives:")
print(test[label_cols].sum())


# Confirm no exact text overlaps between any splits

assert set(train_fit["text_clean"]).isdisjoint(set(val["text_clean"]))
assert set(train_fit["text_clean"]).isdisjoint(set(test["text_clean"]))
assert set(val["text_clean"]).isdisjoint(set(test["text_clean"]))

print("\nNo text overlap between train-fit, validation and test.")


# Save the final three-way split

train_fit.to_csv(
    proc / "train_fit.csv",
    index=False
)

val.to_csv(
    proc / "validation.csv",
    index=False
)

test.to_csv(
    proc / "test.csv",
    index=False
)

print("\nSaved train_fit.csv, validation.csv and test.csv")